In [ ]:
import sys, os
import uuid
sys.path.append(os.path.join(os.path.dirname(os.getcwd())))

from backend.src.graph import create_graph
from backend.src.hitl_handler import handle_hitl

# app = create_graph()
# config = {"configurable": {"thread_id": "hitl-test-1"}}


print("✅ Graph ready")

✅ LLM ready!
✅ Graph ready


In [3]:
# 1) Start a run with a real email so triage_node has mail
dangerous_email = {
    "messages": [],
    "mail": {
        "subject": "Inquiry about pricing",
        "body": (
            "A customer named Sarah (aayushshah90421@gmail.com) "
            "asked for our Q4 price list. Send it to her now."
        ),
    },
}

config = {"configurable": {"thread_id": "hitl-test-1"}}
app = create_graph()

# Run once, even if tools / LLM fail; mail will be in state
try:
    app.invoke(dangerous_email, config)
except Exception:
    pass  # ignore quota errors for this test

# 2) Now override to force a dangerous tool + HITL
state = app.get_state(config)
app.update_state(
    config,
    {
        "tool_name": "send_gmail_reply",
        "tool_args": {
            "to": "aayushshah90421@gmail.com",
            "subject": "Re: Inquiry about pricing",
            "body": "Hi Sarah, ...",
        },
        "hitl": {
            "tool": "send_gmail_reply",
            "args": {
                "to": "aayushshah90421@gmail.com",
                "subject": "Re: Inquiry about pricing",
                "body": "Hi Sarah, ...",
            },
            "proposed_reply": None,
            "triage": state.values.get("triage_category"),
        },
        "hitl_decision": "pending",  # important: pending before user input
    },
)

# 3) Ask user for HITL decision
decision = input("HITL decision (approve/edit/deny): ").strip().lower()
edit_values = None

if decision == "edit":
    # Ask which fields to edit; simple example for 'to'
    new_to = input("New 'to' address (leave blank to keep): ").strip()
    if new_to:
        edit_values = {"to": new_to}

# Apply decision
handle_hitl(app, config, decision=decision, edit_values=edit_values)

# 4) Resume graph to run hitl_checkpoint
app.invoke(None, config)

final_state = app.get_state(config)

print("\n🏁 FINAL STATE:")
print("Final reply:", final_state.values.get("final_reply"))
print("HITL:", final_state.values.get("hitl"))
print("HITL decision:", final_state.values.get("hitl_decision"))
print("Tool name:", final_state.values.get("tool_name"))
print("Tool args:", final_state.values.get("tool_args"))


✅ Triage: respond-act
🔧 Found 1 tools: ['send_gmail_reply']
Email: Sent to aayushshah90421@gmail.com, Subject: Re: Inquiry about pricing
Reply sent to aayushshah90421@gmail.com ID: 19b9cb9ff622fe99
✅ Tool 'None' → No tool to execute.
🔧 Found 0 tools: []
💬 Final reply (no tools)

🏁 FINAL STATE:
Final reply: I have sent the Q4 price list to Sarah.
HITL: None
HITL decision: None
Tool name: None
Tool args: {}
